In [ ]:
from __future__ import annotations

import json
import pathlib
import random
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from kebab.utils.dataset.wikidata import wikidata_utils
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity
from kebab.utils.io_helpers import resolve_path

In [ ]:
exclude_used_entities = True
required_entity_type = ""
exclude_entity_types = ["Q116505632", "Q9332", "Q133500"]

# --- parameters for test dataset ---
# linking_pair_count_limit = 2_000
# linking_entity_count_limit = 10
# linking_property_pattern_count_limit = 20
# linking_property_overlap_limits = {1: 0.35}
# single_class_ratio_limit = 0.76

# --- parameters for training dataset ---
linking_pair_count_limit = 500_000
linking_entity_count_limit = 500
linking_property_pattern_count_limit = 10_000
linking_property_overlap_limits = {1: 0.45}
single_class_ratio_limit = 0.76

output_dir = Path.cwd() / "output"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
use_merged_dataset = True
use_sf_dataset = True
merged_suffix = " Merged" if use_merged_dataset else ""
sf_suffix = " SF" if use_sf_dataset else ""
suffix = f"{sf_suffix}{merged_suffix}"

base_linking_dataset_path = (
    pathlib.Path.cwd() / "Datasets" / "REBEL" / f"Linking{suffix}" / "Base" / "rebel_linking_dataset.jsonl"
)

base_linking_ground_truth_path = (
    pathlib.Path.cwd() / "Datasets" / "REBEL" / f"Linking{suffix}" / "Base" / "rebel_linking_ground_truth.jsonl"
)

base_clustering_dataset_path = (
    pathlib.Path.cwd()
    / "Datasets"
    / "REBEL"
    / f"Linking{suffix}"
    / "Base"
    / "clustering"
    / "rebel_clustering_dataset.jsonl"
)

base_clustering_ground_truth_path = (
    pathlib.Path.cwd()
    / "Datasets"
    / "REBEL"
    / f"Linking{suffix}"
    / "Base"
    / "clustering"
    / "rebel_clustering_ground_truth.jsonl"
)

base_entity_generation_dataset_path = (
    pathlib.Path.cwd()
    / "Datasets"
    / "REBEL"
    / f"Linking{suffix}"
    / "Base"
    / "entity_generation"
    / "rebel_entity_generation_dataset.jsonl"
)

base_fragment_generation_dataset_path = (
    pathlib.Path.cwd()
    / "Datasets"
    / "REBEL"
    / f"Linking{suffix}"
    / "Base"
    / "fragment_generation"
    / "rebel_fragment_generation_dataset.jsonl"
)

In [ ]:
# fragment_id to entity_id map
fragment_to_entity_map_path = (
    pathlib.Path.cwd() / "Datasets" / "REBEL" / f"Linking{suffix}" / "Base" / "rebel_fragment_to_entity_map.jsonl"
)

# Wikidata type hierarchy for filtering by type
type_hierarchy_path = (
    pathlib.Path.cwd() / "Datasets" / "Wikidata" / "Type Hierarchy" / "2025-01-30" / "wikidata_type_hierarchy.jsonl"
)

# entities used in the evaluation datasets
entity_ids_used_for_evaluation = [
    (pathlib.Path.cwd() / "Datasets" / "REBEL" / f"Linking{suffix}" / "Test" / "rebel_linking_used_entities.jsonl"),
    (
        pathlib.Path.cwd()
        / "Datasets"
        / "REBEL"
        / f"Linking{suffix}"
        / "Validation"
        / "rebel_linking_used_entities.jsonl"
    ),
]

In [ ]:
# resolve all paths
base_linking_dataset_path = resolve_path(base_linking_dataset_path)
base_linking_ground_truth_path = resolve_path(base_linking_ground_truth_path)
base_clustering_dataset_path = resolve_path(base_clustering_dataset_path)
base_clustering_ground_truth_path = resolve_path(base_clustering_ground_truth_path)
base_entity_generation_dataset_path = resolve_path(base_entity_generation_dataset_path)
base_fragment_generation_dataset_path = resolve_path(base_fragment_generation_dataset_path)
fragment_to_entity_map_path = resolve_path(fragment_to_entity_map_path)
type_hierarchy_path = resolve_path(type_hierarchy_path)
entity_ids_used_for_evaluation = [resolve_path(p) for p in entity_ids_used_for_evaluation]
output_dir = resolve_path(output_dir)

In [ ]:
disallowed_entity_ids = set()

if exclude_used_entities:
    for entity_ids_path in entity_ids_used_for_evaluation:
        if not entity_ids_path.exists():
            continue

        with open(entity_ids_path, encoding="utf-8") as f:
            for line in f:
                entity_id = json.loads(line)
                disallowed_entity_ids.add(entity_id)

print(f"Disallowed entities that are already used in evaluation dataset: {len(disallowed_entity_ids):,d}")

In [ ]:
if required_entity_type:
    # get the target entity type and its descendants
    get_descendants = False

    graph, type_id_to_node = wikidata_utils.load_type_hierarchy(type_hierarchy_path)
    required_entity_types = (
        wikidata_utils.collect_all_subtypes(graph, type_id_to_node, required_entity_type)
        if get_descendants
        else {required_entity_type}
    )

    required_entity_types = {type_id_to_node[t]["name"] for t in required_entity_types}
else:
    required_entity_types = set()

print(f"Target entity type: {required_entity_type} ({len(required_entity_types):,d} with descendants)")

In [ ]:
if exclude_entity_types:
    # get the target entity type and its descendants
    get_descendants = False
    graph, type_id_to_node = wikidata_utils.load_type_hierarchy(type_hierarchy_path)

    collected_entity_types = set()

    print(f"Excluding entity types: {[type_id_to_node[t]['name'] for t in exclude_entity_types]}")

    for t in exclude_entity_types:
        entity_types = wikidata_utils.collect_all_subtypes(graph, type_id_to_node, t) if get_descendants else {t}

        collected_entity_types.update(entity_types)

    exclude_entity_types = {type_id_to_node[t]["name"] for t in collected_entity_types}
else:
    exclude_entity_types = set()

print(f"Excluded entity types: {len(exclude_entity_types):,d} with descendants ({exclude_entity_types})")

# Linking
---

In [ ]:
# load the ground truth
with open(base_linking_ground_truth_path, encoding="utf-8") as f:
    ground_truth = [json.loads(line) for line in f]

print(f"Loaded {len(ground_truth):,d} ground truth labels")

# load fragment to entity map
fragment_to_entity_map = {}
with open(fragment_to_entity_map_path, encoding="utf-8") as f:
    for line in f:
        f_id, e_id = json.loads(line)
        fragment_to_entity_map[f_id] = e_id

print(f"Loaded {len(fragment_to_entity_map):,d} fragment to entity mappings")

In [ ]:
# load and filter fragments to only include the target entity types
fragments = {}
pairs = []
labels = []
entity_ids = set()
skipped = 0


def add_fragment(fragment: ResolvedWikidataEntity) -> ResolvedWikidataEntity:
    """Add the fragment to the set of known fragments."""
    if fragment.metadata["fragment_id"] not in fragments:
        fragments[fragment.metadata["fragment_id"]] = fragment
        fragment.evidence_map = None
        fragment.source_ids = None

        fragment.entity_id = fragment_to_entity_map[fragment.metadata["fragment_id"]]

    return fragments[fragment.metadata["fragment_id"]]


with open(base_linking_dataset_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        d = json.loads(line)
        left = add_fragment(ResolvedWikidataEntity.from_dict(d[0]))
        right = add_fragment(ResolvedWikidataEntity.from_dict(d[1]))

        if required_entity_types and (
            not set(left.wikidata_type).intersection(required_entity_types)
            or not set(right.wikidata_type).intersection(required_entity_types)
        ):
            skipped += 1
            continue

        if exclude_entity_types and (
            set(left.wikidata_type).intersection(exclude_entity_types)
            or set(right.wikidata_type).intersection(exclude_entity_types)
        ):
            skipped += 1
            continue

        pairs.append((left, right))
        labels.append(ground_truth[i])
        entity_ids.add(left.entity_id)
        entity_ids.add(right.entity_id)

assert len(pairs) == len(labels)

print(f"Skipped {skipped:,d} pairs due to entity type filtering")

print(f"Filtered to {len(pairs):,d} pairs where both entities are of the target types")
print(f"Positive pairs: {sum(labels):,d} ({sum(labels) / len(labels):.2%})")

print(f"Distinct entities: {len(entity_ids):,d} ({len(entity_ids) / len(pairs):.2%})")
with open(output_dir / "rebel_linking_entity_ids_pool.jsonl", "w", encoding="utf-8") as f:
    for entity_id in entity_ids:
        f.write(json.dumps(entity_id) + "\n")

In [ ]:
# sample pairs
sampled_pairs = []
sampled_labels = []

linking_entity_counter = Counter()
property_pattern_counter = Counter()
property_counter = Counter()
type_counter = Counter()
overlap_counter = Counter()
class_counter = Counter()

indices = list(range(len(pairs)))
random.shuffle(indices)
iterations = 0

included_fragment_ids = set()

for i in indices:
    iterations += 1

    if len(sampled_pairs) >= linking_pair_count_limit:
        break

    pair = pairs[i]
    left, right = pair

    if left.entity_id in disallowed_entity_ids or right.entity_id in disallowed_entity_ids:
        continue

    if (
        linking_entity_counter[left.entity_id] >= linking_entity_count_limit
        or linking_entity_counter[right.entity_id] >= linking_entity_count_limit
    ):
        continue

    prop_pattern = tuple(sorted([tuple(sorted(left.properties)), tuple(sorted(right.properties))]))
    if property_pattern_counter[prop_pattern] >= linking_property_pattern_count_limit:
        continue

    # overlap
    left_props = set(left.properties.keys())
    right_props = set(right.properties.keys())
    overlap = left_props.intersection(right_props)
    overlap_num = len(overlap)
    if (
        overlap_num in linking_property_overlap_limits
        and overlap_counter[overlap_num] >= linking_property_overlap_limits[overlap_num] * linking_pair_count_limit
    ):
        continue

    if single_class_ratio_limit and class_counter[labels[i]] > single_class_ratio_limit * linking_pair_count_limit:
        continue

    class_counter[labels[i]] += 1

    overlap_counter[len(overlap)] += 1

    linking_entity_counter[left.entity_id] += 1
    if left.entity_id != right.entity_id:
        linking_entity_counter[right.entity_id] += 1

    property_pattern_counter[prop_pattern] += 1

    for t in set(left.wikidata_type or []).union(right.wikidata_type or []):
        type_counter[t] += 1

    for prop_name in set(left.properties.keys()).union(right.properties.keys()):
        property_counter[prop_name] += 1

    sampled_pairs.append(pairs[i])
    sampled_labels.append(labels[i])
    included_fragment_ids.add(left.metadata["fragment_id"])
    included_fragment_ids.add(right.metadata["fragment_id"])

print(f"Sampled {len(sampled_pairs):,d} pairs containing {len(linking_entity_counter):,d} distinct entities")
print(f"Positive pairs: {sum(sampled_labels):,d} ({sum(sampled_labels) / len(sampled_labels):.2%})")
print(f"Iterations: {iterations:,d}, acceptance rate {len(sampled_pairs) / iterations:.2%}")
print(f"Total included fragments: {len(included_fragment_ids):,d} ({len(included_fragment_ids) / len(fragments):.2%})")

# top entities
print("\nTop entities:")
for i, (k, v) in enumerate(linking_entity_counter.most_common(n=10)):
    print(f"{i}: {k}: {v}")

# top properties
print("\nTop properties:")
for i, (k, v) in enumerate(property_counter.most_common(n=10)):
    print(f"{i}: {k}: {v} ({v / len(sampled_pairs):.2%})")

# top property patterns
print("\nTop property patterns:")
for i, (k, v) in enumerate(property_pattern_counter.most_common(n=10)):
    print(f"{i}: {k}: {v}")

# top types
print("\nTop types:")
for i, (k, v) in enumerate(type_counter.most_common(n=10)):
    print(f"{i}: {k}: {v} ({v / len(sampled_pairs):.2%})")

# top overlaps
print("\nTop overlaps:")
for _, (k, v) in enumerate(sorted(overlap_counter.items(), key=lambda x: x[0])):
    print(f"{k}: {v} ({v / len(sampled_pairs):.2%})")

In [ ]:
data = defaultdict(list)

for pair, label in zip(sampled_pairs, sampled_labels, strict=False):
    left, right = pair
    data["same_entity"].append(label)
    data["same_type"].append(left.wikidata_type == right.wikidata_type)
    data["name_overlap"].append(len(set(left.names).intersection(right.names)) > 0)
    data["lowercase_name_overlap"].append(
        len({n.lower() for n in left.names}.intersection({n.lower() for n in right.names})) > 0
    )

df = pd.DataFrame.from_dict(data)

key = "lowercase_name_overlap"
total = df.shape[0]
positives = df[(df["same_entity"])].shape[0]
negatives = df[(~df["same_entity"])].shape[0]

tp = df[(df["same_entity"]) & (df[key])].shape[0]
fp = df[(~df["same_entity"]) & (df[key])].shape[0]
tn = df[(~df["same_entity"]) & (~df[key])].shape[0]
fn = df[(df["same_entity"]) & (~df[key])].shape[0]

assert tp + fp + tn + fn == total

precision = tp / (tp + fp)
recall = tp / (tp + fn)
ppv = precision
npv = tn / (tn + fn)

print("Ground truth:")
print(f"Positives = {positives:,d} ({positives / total:.1%})")
print(f"Negatives = {negatives:,d} ({negatives / total:.1%})")
print()
print("Linking by name:")
print(f"True positives = {tp:,d} ({tp / total:.1%})")
print(f"False positives = {fp:,d} ({fp / total:.1%})")
print(f"True negatives = {tn:,d} ({tn / total:.1%})")
print(f"False negatives = {fn:,d} ({fn / total:.1%})")
print()
print(f"Precision = {precision:.4f}")
print(f"Recall = {recall:.4f}")
print()
print(f"P(positive | predict positive) = {ppv:.4f}")
print(f"P(negative | predict negative) = {npv:.4f}")

In [ ]:
size = len(sampled_pairs)

# write the sampled pairs and labels to files
with open(output_dir / "rebel_linking_dataset.jsonl", "w", encoding="utf-8") as f:
    for pair in sampled_pairs:
        d = [
            pair[0].to_dict(minimal_repr=True),
            pair[1].to_dict(minimal_repr=True),
        ]
        f.write(json.dumps(d) + "\n")

with open(output_dir / "rebel_linking_ground_truth.jsonl", "w", encoding="utf-8") as f:
    for label in sampled_labels:
        f.write(json.dumps(label) + "\n")

with open(output_dir / "rebel_linking_used_entities.jsonl", "w", encoding="utf-8") as f:
    for entity_id in linking_entity_counter.keys():
        f.write(json.dumps(entity_id) + "\n")

# Clustering
---

In [ ]:
fragments_to_include = set()
for pair in sampled_pairs:
    fragments_to_include.add(pair[0].metadata["fragment_id"])
    fragments_to_include.add(pair[1].metadata["fragment_id"])

entities_to_include = set(linking_entity_counter)

In [ ]:
clustering_entities = defaultdict(list)

with (
    open(base_clustering_dataset_path, encoding="utf-8") as f_ds,
    open(base_clustering_ground_truth_path, encoding="utf-8") as f_gt,
):
    for line_ds, line_gt in zip(f_ds, f_gt, strict=False):
        fragment = ResolvedWikidataEntity.from_dict(json.loads(line_ds))
        entity_id = json.loads(line_gt)

        if entity_id not in entities_to_include:
            continue

        clustering_entities[entity_id].append(fragment)

# clustering_entities = dict(sorted(clustering_entities.items(), key=lambda kvp: -len(kvp[1])))

In [ ]:
clustering_fragment_count_min = 3
clustering_entity_count_limit = 100 if len(entities_to_include) < 10_000 else 10_000_000
clustering_fragment_count_limit = 50 if len(entities_to_include) < 10_000 else 1000
clustering_include_all_linking_fragments = True

import random


ent_with_fragments = list(clustering_entities.items())
random.shuffle(ent_with_fragments)

# filter the clustering dataset and ground truth to only include the sampled entities
clustering_entity_counter = Counter()

# size = clustering_entity_count_limit if clustering_entity_count_limit > 0 else len(allowed_entity_ids)
c_output_dir = output_dir / "clustering"
c_output_dir.mkdir(parents=True, exist_ok=True)

with (
    open(c_output_dir / "rebel_clustering_dataset.jsonl", "w", encoding="utf-8") as f_ds_out,
    open(c_output_dir / "rebel_clustering_ground_truth.jsonl", "w", encoding="utf-8") as f_gt_out,
):
    for entity_id, fragments in ent_with_fragments:
        if len(fragments) < clustering_fragment_count_min:
            continue

        if (
            0 < clustering_entity_count_limit <= len(clustering_entity_counter)
            and entity_id not in clustering_entity_counter
        ):
            continue

        for fragment in sorted(fragments, key=lambda f: f.metadata["fragment_id"] not in included_fragment_ids)[
            :clustering_fragment_count_limit
        ]:
            record = fragment.without_entity_id().to_dict(minimal_repr=True)
            f_ds_out.write(json.dumps(record) + "\n")
            f_gt_out.write(entity_id)

            clustering_entity_counter[entity_id] += 1

print(
    f"Filtered to {sum(clustering_entity_counter.values()):,d} fragments for {len(clustering_entity_counter):,d} entities, target = {clustering_entity_count_limit:,d} entities"
)

print("\nTop entities:")
for i, (k, v) in enumerate(clustering_entity_counter.most_common(n=100)):
    print(f"{i}: {k}: {v}")

# Entity Generation
---

In [ ]:
e_output_dir = output_dir / "entity_generation"
e_output_dir.mkdir(parents=True, exist_ok=True)

include_all = len(entities_to_include) > 10_000  # means we are building the training dataset
count = 0

with (
    open(base_entity_generation_dataset_path, encoding="utf-8") as f_ds,
    open(e_output_dir / "rebel_entity_generation_dataset.jsonl", "w", encoding="utf-8") as f_ds_out,
):
    for line in f_ds:
        fragment = ResolvedWikidataEntity.from_dict(json.loads(line))
        entity_id = fragment.entity_id

        if not include_all and entity_id not in entities_to_include:
            continue

        if entity_id in disallowed_entity_ids:
            continue

        f_ds_out.write(json.dumps(fragment.without_entity_id().to_dict(minimal_repr=True)) + "\n")
        count += 1

print(f"Filtered to {count:,d} entities for entity generation dataset")

# Fragment Generation
---

In [ ]:
f_output_dir = output_dir / "fragment_generation"
f_output_dir.mkdir(parents=True, exist_ok=True)

include_all = len(entities_to_include) > 10_000  # means we are building the training dataset
count = 0

with (
    open(base_fragment_generation_dataset_path, encoding="utf-8") as f_ds,
    open(f_output_dir / "rebel_fragment_generation_dataset.jsonl", "w", encoding="utf-8") as f_ds_out,
):
    for line in f_ds:
        fragment = ResolvedWikidataEntity.from_dict(json.loads(line))
        entity_id = fragment.entity_id

        if not include_all and entity_id not in entities_to_include:
            continue

        if entity_id in disallowed_entity_ids:
            continue

        f_ds_out.write(json.dumps(fragment.without_entity_id().to_dict(minimal_repr=True)) + "\n")
        count += 1

print(f"Filtered to {count:,d} entities for fragment generation dataset")